# Classificação de imagens RGB com CNNs usando CIFAR-10

Aprendizagem Profunda — Mestrado em Engenharia Informática / IA / ECD  
Universidade do Minho, 2025/2026

## T1 – Download e extração do dataset CIFAR-10

In [ ]:
import os

if not os.path.exists('cifar'):
    !wget http://pjreddie.com/media/files/cifar.tgz
    !tar xzf cifar.tgz
    print('Dataset downloaded and extracted.')
else:
    print('Dataset already exists, skipping download.')

## T2 – Instalação de dependências

In [ ]:
# Check for CUDA / nvcc
!nvcc --version 2>/dev/null || echo 'nvcc not found (CPU-only environment)'
!nvidia-smi 2>/dev/null || echo 'nvidia-smi not found'

In [ ]:
!pip install -q torch torchvision livelossplot gdown matplotlib numpy

## T3 – Definir batch_size

In [ ]:
BATCH_SIZE = 128
print(f'batch_size = {BATCH_SIZE}')

## T4 – Preparação dos dados

### T4.1 – Lista de classes do dataset

In [ ]:
# CIFAR-10 classes (order matches the dataset label indices)
CLASSES = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck',
]
print(f'Classes ({len(CLASSES)}): {CLASSES}')

### T4.2 & T4.3 – Pré-processamento: normalização e conversão para CHW

In [ ]:
import torch
import torchvision.transforms as transforms

# Mean and std computed over the CIFAR-10 training set (per channel)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

# Training transforms: random augmentations + normalisation
# ToTensor() converts HWC uint8 [0,255] → CHW float32 [0,1]  (T4.3)
# Normalize() applies per-channel mean/std normalisation       (T4.2)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),                          # HWC → CHW, [0,1]
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Validation / test transforms: only normalisation (no augmentation)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

print('Transforms defined.')
print('  train_transform:', train_transform)
print('  eval_transform :', eval_transform)

### T4.4 – Leitura do dataset e preparação dos DataLoaders (holdout)

In [ ]:
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split


class CIFAR10PJReddie(Dataset):
    """Loads the pjreddie CIFAR-10 dataset.

    The tgz archive produces a ``cifar/`` folder with:
      cifar/train/<idx>_<classname>.png
      cifar/test/<idx>_<classname>.png
      cifar/labels.txt
    """

    def __init__(self, root: str, split: str = 'train', transform=None):
        assert split in ('train', 'test')
        self.transform = transform
        self.samples = []  # list of (path, class_index)

        split_dir = os.path.join(root, split)
        for img_path in sorted(glob.glob(os.path.join(split_dir, '*.png'))):
            # filename format: "<idx>_<classname>.png"
            basename = os.path.splitext(os.path.basename(img_path))[0]
            class_name = '_'.join(basename.split('_')[1:])  # handles multi-word names
            if class_name not in CLASSES:
                continue
            class_idx = CLASSES.index(class_name)
            self.samples.append((img_path, class_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


CIFAR_ROOT = 'cifar'

# Full training set (50 000 images)
full_train_dataset = CIFAR10PJReddie(CIFAR_ROOT, split='train', transform=train_transform)

# Holdout split: 80 % train  /  20 % validation
VAL_RATIO   = 0.2
n_total     = len(full_train_dataset)
n_val       = int(n_total * VAL_RATIO)
n_train     = n_total - n_val

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)

# Apply eval transform to the validation subset
# We wrap val_dataset so it uses eval_transform instead of train_transform
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        # Retrieve raw PIL image by bypassing the original transform
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


val_dataset = TransformSubset(val_dataset, eval_transform)

# Test set (10 000 images)
test_dataset = CIFAR10PJReddie(CIFAR_ROOT, split='test', transform=eval_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples      : {len(train_dataset):>6}')
print(f'Validation samples : {len(val_dataset):>6}')
print(f'Test samples       : {len(test_dataset):>6}')
print(f'Train batches      : {len(train_loader):>6}')
print(f'Val batches        : {len(val_loader):>6}')
print(f'Test batches       : {len(test_loader):>6}')

## T5 – Visualização dos dados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Reverse normalisation for display
def denormalize(tensor, mean=CIFAR10_MEAN, std=CIFAR10_STD):
    """Convert a normalised CHW tensor back to a displayable HWC numpy array."""
    t = tensor.clone()
    for c, (m, s) in enumerate(zip(mean, std)):
        t[c] = t[c] * s + m
    return t.permute(1, 2, 0).numpy().clip(0, 1)


# --- Dataset metrics ---
print('=== Dataset metrics ===')
print(f'  Total images : {len(full_train_dataset) + len(test_dataset)}')
print(f'  Training     : {len(train_dataset)}')
print(f'  Validation   : {len(val_dataset)}')
print(f'  Test         : {len(test_dataset)}')
print(f'  Classes      : {len(CLASSES)}')
print(f'  Image size   : 32 × 32 × 3 (RGB)')
print(f'  Batch size   : {BATCH_SIZE}')

# --- Visualise one training batch ---
images, labels = next(iter(train_loader))
print(f'\nBatch tensor shape : {images.shape}  (N × C × H × W)')

n_show = 16
fig, axes = plt.subplots(2, n_show // 2, figsize=(n_show, 5))
axes = axes.flatten()

for i in range(n_show):
    img_np = denormalize(images[i])
    axes[i].imshow(img_np)
    axes[i].set_title(CLASSES[labels[i].item()], fontsize=9)
    axes[i].axis('off')

fig.suptitle('Sample training batch (16 images)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## T6 – Verificação do balanceamento do dataset

In [ ]:
from collections import Counter


def class_distribution(dataset, name: str):
    """Count samples per class and print a summary table."""
    if hasattr(dataset, 'samples'):
        # CIFAR10PJReddie
        all_labels = [label for _, label in dataset.samples]
    elif hasattr(dataset, 'subset'):
        # TransformSubset (validation)
        all_labels = [dataset.subset.dataset.samples[dataset.subset.indices[i]][1]
                      for i in range(len(dataset))]
    else:
        # random_split Subset — access via underlying dataset indices
        all_labels = [dataset.dataset.samples[i][1] for i in dataset.indices]

    counts = Counter(all_labels)
    total  = sum(counts.values())

    print(f'\n=== {name} ({total} samples) ===')
    print(f'  {"Class":<12} {"Count":>6}  {"Share":>7}')
    print('  ' + '-' * 28)
    for idx, cls in enumerate(CLASSES):
        n   = counts.get(idx, 0)
        pct = 100 * n / total if total > 0 else 0
        print(f'  {cls:<12} {n:>6}  {pct:>6.2f} %')

    return counts


train_counts = class_distribution(train_dataset, 'Training set')
val_counts   = class_distribution(val_dataset,   'Validation set')
test_counts  = class_distribution(test_dataset,  'Test set')

In [ ]:
# Visual bar chart of class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)

split_info = [
    ('Training set',    train_counts),
    ('Validation set',  val_counts),
    ('Test set',        test_counts),
]

for ax, (title, counts) in zip(axes, split_info):
    values = [counts.get(i, 0) for i in range(len(CLASSES))]
    bars = ax.bar(CLASSES, values, color='steelblue', edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of samples')
    ax.tick_params(axis='x', rotation=45)
    # Annotate bar heights
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                str(v), ha='center', va='bottom', fontsize=7)

    # Reference line for a perfectly balanced dataset
    expected = sum(values) / len(CLASSES)
    ax.axhline(expected, color='red', linestyle='--', linewidth=1.2,
               label=f'Expected ({int(expected)})')
    ax.legend(fontsize=8)

fig.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nConclusion: CIFAR-10 is a perfectly balanced dataset — each class '
      'contains the same number of samples in every split.')